In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd

In [2]:
mammals = pd.read_csv('data/Mammals.csv')
reptiles = pd.read_csv('data/Reptiles.csv')
amphibians = pd.read_csv('data/Amphibians.csv')
birds = pd.read_csv('data/Birds.csv')

In [4]:
vertebrate_iucn = gpd.read_file('data/congo_fauna_biodiversity.gpkg')

In [43]:
group = ['Amphibian', 'Bird', 'Mammal', 'Reptile']

for g in group:
    subset = vertebrate_iucn.loc[vertebrate_iucn['group']==g]
    trophics=[]
    if g=='Amphibian':
        for A in subset.sci_name:
            if A in amphibians['Best_guess_binomial'].values:
                trophic = amphibians['Trophic_level'].loc[amphibians['Best_guess_binomial']==A] 
                trophics.append(trophic.iloc[0])
            else:
                trophic = ''
                trophics.append(trophic)
        subset['trophic_level'] = trophics
        subset.to_csv('amphibian_trophic_level_congo_basin.csv', index=False)
    elif g=='Bird':
        for B in subset.sci_name:
            if B in birds['Best_guess_binomial'].values:
                trophic = birds['Trophic_level'].loc[birds['Best_guess_binomial']==B] 
                trophics.append(trophic.iloc[0])
            else:
                trophic = ''
                trophics.append(trophic)
        subset['trophic_level'] = trophics
        subset.to_csv('birds_trophic_level_congo_basin.csv', index=False)
    elif g=='Mammal':
        for M in subset.sci_name:
            if M in mammals['Best_guess_binomial'].values:
                trophic = mammals['Trophic_level'].loc[mammals['Best_guess_binomial']==M] 
                trophics.append(trophic.iloc[0])
            else:
                trophic = ''
                trophics.append(trophic)
        subset['trophic_level'] = trophics
        subset.to_csv('mammals_trophic_level_congo_basin.csv', index=False)
    else:
        for R in subset.sci_name:
            if R in reptiles['Best_guess_binomial'].values:
                trophic = reptiles['Trophic_level'].loc[reptiles['Best_guess_binomial']==R]
                trophics.append(trophic.iloc[0])
            else:
                trophic = ''
                trophics.append(trophic)
        subset['trophic_level'] = trophics
        subset.to_csv('reptiles_trophic_level_congo_basin.csv', index=False)

CONSTRUCT PREDICTORS MATRIX

In [45]:
topography = gpd.read_file('data/grid_topography_variables.gpkg')
hydrology = gpd.read_file('data/congo_basin_hydrology_grid.gpkg')
mosaic_vegetation = gpd.read_file('data/frank_white_mosaic_vegetation_by_grid_species.gpkg')
major_vegetation = gpd.read_file('data/frank_white_major_vegetation_by_grid_species.gpkg')
paleoclimate_BI01 = gpd.read_file('data/BIO1_temporal_standard_deviation.gpkg')
paleoclimate_BI012 = gpd.read_file('data/BIO12_temporal_standard_deviation.gpkg')
refugia_distance = gpd.read_file('data/grid_refugia_distance.gpkg')
tree_cover_2000 = gpd.read_file('data/Tree_cover2000_Congo_Basin.gpkg')
forest_canopy_2019 = gpd.read_file('data/Forest_canopy_height_Congo_Basin_2019.gpkg')
forest_loss_2001_2020 = gpd.read_file('data/forest_loss_2001_2020.gpkg')
mean_soil_PH_water = gpd.read_file('data/Mean_soil_pH_water_depth_0_5_cm_Congo_Basin.gpkg')
soc = gpd.read_file('data/soc_0_5cm_mean.gpkg')
cec = gpd.read_file('data/cec_0_5cm_mean.gpkg')
clay = gpd.read_file('data/clay_0_5cm_mean.gpkg')

In [94]:
tree_cover_2000

,grid,weight_list,WE,richness,CWE,tree_cover_mean,tree_cover_stdev,geometry
0,308,"[6.941070313042272e-05, 3.8892345986309894e-05...",0.083605,509,0.000164,2.245411,4.639130,"MULTIPOLYGON (((7.97286 5.20776, 7.96912 5.212..."
1,309,"[6.941070313042272e-05, 3.8892345986309894e-05...",0.083577,509,0.000164,6.760294,6.364646,"MULTIPOLYGON (((7.97019 5.25216, 7.9713 5.2538..."
2,318,"[6.941070313042272e-05, 3.8892345986309894e-05...",0.079305,509,0.000156,14.407789,4.065166,"MULTIPOLYGON (((7.97236 5.73213, 7.96792 5.745..."
3,319,"[6.941070313042272e-05, 3.8892345986309894e-05...",0.079407,514,0.000154,13.413355,11.528500,"MULTIPOLYGON (((7.96499 5.75398, 7.95362 5.772..."
4,320,"[6.941070313042272e-05, 3.417751802864076e-05,...",0.079317,507,0.000156,15.931753,11.134698,"MULTIPOLYGON (((7.93881 5.79709, 7.93473 5.805..."
...,...,...,...,...,...,...,...,...
114942,177138,"[6.157635467980295e-05, 9.470414425335253e-06,...",0.236881,666,0.000356,29.973684,11.300031,"MULTIPOLYGON (((32.05357 1.99546, 32.06525 1.9..."
114943,177503,"[6.157635467980295e-05, 9.470414425335253e-06,...",0.185886,659,0.000282,28.816572,1.016684,"MULTIPOLYGON (((32.07396 1.94478, 32.07396 1.9..."
114944,177504,"[6.157635467980295e-05, 9.470414425335253e-06,...",0.243488,666,0.000366,25.201028,8.183261,"MULTIPOLYGON (((32.12164 1.99455, 32.10951 1.9..."
114945,177505,"[6.157635467980295e-05, 9.470414425335253e-06,...",0.243263,665,0.000366,22.685388,7.191245,"MULTIPOLYGON (((32.07656 1.99885, 32.09024 2.0..."


In [88]:
#mosaic_vegetation.rename(columns={'vegetation_richness': 'mosaic_vegetation_richness', 'vegetation_shannon': 'mosaic_vegetation_shannon', 'vegetation_evenness': 'mosaic_vegetation_evenness' }, inplace=True)

river_selected = (hydrology.sort_values(["grid", "ORD_STRA", "DIS_AV_CMS"],ascending=[True, False, False]).drop_duplicates("grid"))

river_selected

,Unnamed: 0,grid,distance_km,river_direction,ORD_STRA,UPLAND_SKM,LENGTH_KM,DIS_AV_CMS,species,weight_list,WE,richness,CWE,geometry
0,308,308,2.092740,E,5,1887.599976,0.92,94.171997,"['Afrixalus weidholzi', 'Opisthothylax immacul...",0.000015,0.000015,509,2.966161e-08,"MULTIPOLYGON (((7.97286 5.20776, 7.96912 5.212..."
1,309,309,3.563950,E,7,49258.898438,8.56,2179.149902,"['Afrixalus weidholzi', 'Opisthothylax immacul...",0.000015,0.000015,509,2.966161e-08,"MULTIPOLYGON (((7.97019 5.25216, 7.9713 5.2538..."
2,318,318,3.112026,E,7,47122.601562,5.01,2016.026978,"['Afrixalus weidholzi', 'Opisthothylax immacul...",0.000015,0.000015,509,2.966161e-08,"MULTIPOLYGON (((7.97236 5.73213, 7.96792 5.745..."
3,319,319,1.295206,E,7,46968.601562,2.89,2008.812012,"['Afrixalus weidholzi', 'Opisthothylax immacul...",0.000015,0.000015,514,2.937307e-08,"MULTIPOLYGON (((7.96499 5.75398, 7.95362 5.772..."
5,320,320,1.854625,E,7,46951.300781,2.89,2008.005005,"['Afrixalus weidholzi', 'Ptychadena aequiplica...",0.000015,0.000015,507,2.977862e-08,"MULTIPOLYGON (((7.93881 5.79709, 7.93473 5.805..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137790,177138,177138,188.252867,W,5,3208.800049,2.89,21.097000,"['Hyperolius balfouri', 'Afrixalus quadrivitta...",0.000016,0.000016,666,2.468357e-08,"MULTIPOLYGON (((32.05357 1.99546, 32.06525 1.9..."
137791,177503,177503,185.634455,W,5,2386.399902,1.12,12.119000,"['Hyperolius balfouri', 'Afrixalus quadrivitta...",0.000016,0.000016,659,2.494576e-08,"MULTIPOLYGON (((32.07396 1.94478, 32.07396 1.9..."
137792,177504,177504,189.307182,W,5,2386.399902,1.12,12.119000,"['Hyperolius balfouri', 'Afrixalus quadrivitta...",0.000016,0.000016,666,2.468357e-08,"MULTIPOLYGON (((32.12164 1.99455, 32.10951 1.9..."
137793,177505,177505,193.069530,W,5,2386.399902,1.12,12.119000,"['Hyperolius balfouri', 'Afrixalus quadrivitta...",0.000016,0.000016,665,2.472069e-08,"MULTIPOLYGON (((32.07656 1.99885, 32.09024 2.0..."


In [3]:
mean_soil_PH_water = gpd.read_file('data/Mean_soil_pH_water_depth_0_5_cm_Congo_Basin.gpkg')

In [4]:
mean_soil_PH_water

,grid,weight_list,WE,richness,CWE,_mean,_stdev,geometry
0,308,"[6.941070313042272e-05, 3.8892345986309894e-05...",0.083605,509,0.000164,53.639026,0.767360,"MULTIPOLYGON (((7.97286 5.20776, 7.96912 5.212..."
1,309,"[6.941070313042272e-05, 3.8892345986309894e-05...",0.083577,509,0.000164,53.000000,0.000000,"MULTIPOLYGON (((7.97019 5.25216, 7.9713 5.2538..."
2,318,"[6.941070313042272e-05, 3.8892345986309894e-05...",0.079305,509,0.000156,54.750000,0.433013,"MULTIPOLYGON (((7.97236 5.73213, 7.96792 5.745..."
3,319,"[6.941070313042272e-05, 3.8892345986309894e-05...",0.079407,514,0.000154,54.388350,0.543863,"MULTIPOLYGON (((7.96499 5.75398, 7.95362 5.772..."
4,320,"[6.941070313042272e-05, 3.417751802864076e-05,...",0.079317,507,0.000156,54.528302,0.650218,"MULTIPOLYGON (((7.93881 5.79709, 7.93473 5.805..."
...,...,...,...,...,...,...,...,...
114942,177138,"[6.157635467980295e-05, 9.470414425335253e-06,...",0.236881,666,0.000356,61.333333,0.471405,"MULTIPOLYGON (((32.05357 1.99546, 32.06525 1.9..."
114943,177503,"[6.157635467980295e-05, 9.470414425335253e-06,...",0.185886,659,0.000282,61.000000,0.000000,"MULTIPOLYGON (((32.07396 1.94478, 32.07396 1.9..."
114944,177504,"[6.157635467980295e-05, 9.470414425335253e-06,...",0.243488,666,0.000366,60.955801,0.443781,"MULTIPOLYGON (((32.12164 1.99455, 32.10951 1.9..."
114945,177505,"[6.157635467980295e-05, 9.470414425335253e-06,...",0.243263,665,0.000366,60.879699,0.368651,"MULTIPOLYGON (((32.07656 1.99885, 32.09024 2.0..."


In [5]:
mean_soil_PH_water.rename(columns={'_mean': 'ph_mean', '_stdev': 'ph_stdev' }, inplace=True)

In [6]:
pred = pd.read_csv('data/predictors.csv')

In [8]:
pred["ph_mean"] = mean_soil_PH_water["ph_mean"]
pred["ph_stdev"] = mean_soil_PH_water["ph_stdev"]

In [49]:
predictors = pd.merge(predictors, mean_soil_PH_water[['grid','ph_mean','clay_stdev']], on='grid', how='inner')

In [89]:
river_selected.columns

Index(['Unnamed: 0', 'grid', 'distance_km', 'river_direction', 'ORD_STRA',
       'UPLAND_SKM', 'LENGTH_KM', 'DIS_AV_CMS', 'species', 'weight_list', 'WE',
       'richness', 'CWE', 'geometry'],
      dtype='str')

In [9]:
pred

,grid,clay_mean,clay_stdev,cec_mean,cec_stdev,soc_mean,soc_stdev,ph_mean,ph_stdev,forest_loss_mean,...,roughness_mean,roughness_std,hillshade_mean,hillshade_std,distance_km,river_direction,ORD_STRA,UPLAND_SKM,LENGTH_KM,DIS_AV_CMS
0,308,142.316300,12.164825,118.115925,4.746745,418.134629,15.401421,53.639026,0.767360,0.000565,...,0.187585,1.236777,176.885475,25.039922,2.092740,E,5,1887.599976,0.92,94.171997
1,309,144.500000,7.500000,108.000000,5.000000,377.502763,31.388362,53.000000,0.000000,0.000000,...,0.082371,0.872032,179.639168,14.766958,3.563950,E,7,49258.898438,8.56,2179.149902
2,318,203.500000,4.031129,98.250000,4.815340,325.200000,30.954160,54.750000,0.433013,0.002291,...,0.178802,1.396401,177.894848,21.892397,3.112026,E,7,47122.601562,5.01,2016.026978
3,319,192.320388,18.484427,99.495146,10.391253,330.451923,44.157254,54.388350,0.543863,0.024812,...,1.537089,2.991020,144.486468,66.984168,1.295206,E,7,46968.601562,2.89,2008.812012
4,320,169.535849,17.900765,92.811321,10.879194,298.584906,40.068260,54.528302,0.650218,0.015030,...,3.886034,5.168086,105.348665,78.693886,1.854625,E,7,46951.300781,2.89,2008.005005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114942,177138,345.000000,8.103497,207.166667,2.409472,371.166667,9.441339,61.333333,0.471405,0.047847,...,5.815370,80.431084,180.061790,10.832829,188.252867,W,5,3208.800049,2.89,21.097000
114943,177503,334.000299,0.706896,198.999403,1.413791,383.990744,21.913766,61.000000,0.000000,0.000000,...,0.204414,15.019895,180.966667,2.449263,185.634455,W,5,2386.399902,1.12,12.119000
114944,177504,345.618785,9.162347,219.988950,10.245058,361.966667,30.489689,60.955801,0.443781,0.010151,...,25.579352,161.853884,120.677716,81.114092,189.307182,W,5,2386.399902,1.12,12.119000
114945,177505,341.766917,8.918922,221.601504,7.012277,385.421053,30.200635,60.879699,0.368651,0.007284,...,20.296420,145.652046,146.817037,66.169742,193.069530,W,5,2386.399902,1.12,12.119000


In [10]:
pred.to_csv('data/predictors.csv', index=False)

In [95]:
predictors.columns

Index(['grid', 'clay_mean', 'clay_stdev', 'cec_mean', 'cec_stdev', 'soc_mean',
       'soc_stdev', 'ph_mean', 'ph_stdev', 'forest_loss_mean',
       'forest_canopy_mean', 'forest_canopy_stdev', 'tree_cover_mean',
       'tree_cover_stdev', 'refugia_dist_km', 'BIO12_temporal_mean',
       'BIO12_temporal_SD', 'BIO1_temporal_mean', 'BIO1_temporal_SD',
       'ALTIMONTANE VEGETATION', 'AZONAL VEGETATION',
       'BUSHLAND AND THICKET MOSAICS', 'FOREST',
       'FOREST TRANSITIONS AND MOSAICS', 'GRASSLAND', 'OUTSIDE AREA',
       'SECONDARY WOODED GRASSLAND', 'SEMI-DESERT VEGETATION', 'WATER_x',
       'WOODLAND', 'WOODLAND MOSAICS AND TRANSITIONS',
       'major_vegetation_richness', 'major_vegetation_shannon',
       'major_vegetation_evenness', 'APSWG', 'ESGKS', 'GCRF-D', 'ITA', 'LRF-W',
       'MANG', 'MOS-1A2', 'MOS-8A1A', 'MOS-EAB-AWG', 'MOS-LRF-IW-SG',
       'MOS-LRF-SG', 'MOS-WZW-SG', 'OUTSIDE', 'SDGS', 'SW-I', 'SWAMP', 'TRF',
       'UMV', 'UW', 'WACM', 'WATER_y', 'WZMW', 'mosaic